In [2]:
import pandas as pd

MF_RF_DATA = "./market_Factor_risk_Free.csv"
STOCK_DATA = "./Stocks_data.csv"

In [18]:
factor_df = pd.read_csv(MF_RF_DATA)
factor_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3469 entries, 0 to 3468
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    3469 non-null   str    
 1   MF      3469 non-null   float64
 2   RF      3469 non-null   float64
dtypes: float64(2), str(1)
memory usage: 81.4 KB


In [19]:
stock_df = pd.read_csv(STOCK_DATA)
stock_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 52 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Dates               3652 non-null   str    
 1   EIM IS Equity       3652 non-null   float64
 2   AXSB IS Equity      3652 non-null   float64
 3   BJFIN IS Equity     3652 non-null   float64
 4   SBILIFE IS Equity   1371 non-null   float64
 5   RELIANCE IS Equity  3652 non-null   float64
 6   ITC IS Equity       3652 non-null   float64
 7   MM IS Equity        3652 non-null   float64
 8   SBIN IS Equity      3652 non-null   float64
 9   HNDL IS Equity      3652 non-null   float64
 10  TATA IS Equity      3652 non-null   float64
 11  SUNP IS Equity      3652 non-null   float64
 12  TTAN IS Equity      3652 non-null   float64
 13  HDFCB IS Equity     3652 non-null   float64
 14  BPCL IS Equity      3652 non-null   float64
 15  CIPLA IS Equity     3652 non-null   float64
 16  APNT IS Equity   

## Data cleaning

In [40]:
stock_df["Dates"] = pd.to_datetime(stock_df["Dates"], format="%d-%m-%Y").dt.date
factor_df["Date"] = pd.to_datetime(factor_df["Date"], format="%d-%m-%Y").dt.date

In [41]:
factor_df["MF_decimal"] = factor_df["MF"] / 100

## Understanding Data

### Checking for date duplicates

In [ ]:
# No duplicate dates recorded
print(
    any(stock_df["Dates"].duplicated()),
    any(factor_df["Date"].duplicated())
)

False False


### Finding uncommon dates (to remove them)

In [ ]:
# Different shapes
print(factor_df.shape[0], stock_df.shape[0])

3469 3652


In [68]:
stock_dates = set(stock_df["Dates"])
factor_dates = set(factor_df["Date"])

extra_stock_dates = stock_dates - factor_dates
extra_factor_dates = factor_dates - stock_dates
unrequired_dates = extra_factor_dates.union(extra_stock_dates)

print(
    len(unrequired_dates),
    2 * len(stock_dates.union(factor_dates)) - (stock_df.shape[0] + factor_df.shape[0])
)

211 211


In [77]:
stock_df = stock_df[~stock_df["Dates"].isin(extra_stock_dates)]
factor_df = factor_df[~factor_df["Date"].isin(extra_factor_dates)]

print(len(stock_df), len(factor_df))

3455 3455


### Missing Values

In [78]:
# Missing stock values
stock_mask = (stock_df.count() != stock_df.shape[0])
stock_partial_columns = stock_mask[stock_mask].index.to_list()

In [79]:
start_dates = {}
for column in stock_partial_columns:
    index = stock_df[column].first_valid_index()
    start_dates[column] = stock_df.loc[index, "Dates"]

start_dates

{'SBILIFE IS Equity': datetime.date(2017, 9, 29),
 'NEST IS Equity': datetime.date(2010, 1, 8),
 'LTIM IS Equity': datetime.date(2016, 7, 20),
 'COAL IS Equity': datetime.date(2010, 11, 3),
 'HDFCLIFE IS Equity': datetime.date(2017, 11, 16)}

In [80]:
# No missing factor values
factor_mask = (factor_df.count() != factor_df.shape[0])
any(factor_mask)

False

### Finishing EDA

In [82]:
stock_df.to_csv("stock_data_cleaned.csv", index=False)
factor_df.to_csv("factor_data_cleaned.csv", index=False)

In [ ]:
from datetime import datetime

for column in stock_df.drop(columns=['Dates']).columns:
    if column not in start_dates:
        start_dates[column] = "2009-01-01"
    else:
        start_dates[column] = start_dates[column].strftime("%y-%m-%d")

start_dates

AttributeError: 'str' object has no attribute 'strftime'

In [98]:
stock_df.columns

Index(['Dates', 'EIM IS Equity', 'AXSB IS Equity', 'BJFIN IS Equity',
       'SBILIFE IS Equity', 'RELIANCE IS Equity', 'ITC IS Equity',
       'MM IS Equity', 'SBIN IS Equity', 'HNDL IS Equity', 'TATA IS Equity',
       'SUNP IS Equity', 'TTAN IS Equity', 'HDFCB IS Equity', 'BPCL IS Equity',
       'CIPLA IS Equity', 'APNT IS Equity', 'ONGC IS Equity', 'JSTL IS Equity',
       'WPRO IS Equity', 'DRRD IS Equity', 'HMCL IS Equity', 'HUVR IS Equity',
       'INFO IS Equity', 'KMB IS Equity', 'LT IS Equity', 'MSIL IS Equity',
       'ADE IS Equity', 'NEST IS Equity', 'NTPC IS Equity', 'LTIM IS Equity',
       'IIB IS Equity', 'APHS IS Equity', 'HCLT IS Equity', 'PWGR IS Equity',
       'BAF IS Equity', 'COAL IS Equity', 'ICICIBC IS Equity',
       'BHARTI IS Equity', 'ADSEZ IS Equity', 'TCS IS Equity',
       'TTMT IS Equity', 'HDFCLIFE IS Equity', 'DIVI IS Equity',
       'UTCEM IS Equity', 'UPLL IS Equity', 'TECHM IS Equity',
       'BJAUT IS Equity', 'BRIT IS Equity', 'GRASIM IS Equity

In [91]:
import json

with open("stock_start_dates.json", 'w') as f:
    json.dump(start_dates, f)